In [ ]:
import pandas as pd
import plotly.express as px

file_path = '/content/Acceptance - Trust Scale (Responses) - Form responses 1.csv'
df = pd.read_csv(file_path)
df.drop(columns=['Timestamp'], inplace=True)
df.head()

,Participant ID,Choose the Experiment Condition,Personality Trait,I find such a system...,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,I trust that the autonomous vehicle will operate safely and effectively under various conditions [Please click one that applies]
0,1,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,2,Agree
1,1,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,3,Agree
2,2,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree
3,2,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree
4,3,A,High Neuroticism - Low Openness,1,2,4,2,2,5,2,5,2,Agree


In [ ]:
df.rename(columns={
    'Participant ID': 'participant_id',
    'Choose the Experiment Condition': 'experiment_condition',
    'Personality Trait': 'personality_trait',
    'I find such a system...': 'useful/useless',
    'Unnamed: 5': 'pleasant/unpleasant',
    'Unnamed: 6': 'bad/good',
    'Unnamed: 7': 'nice/annoying',
    'Unnamed: 8': 'effective/superfluous',
    'Unnamed: 9': 'irritating/likeable',
    'Unnamed: 10': 'assisting/worthless',
    'Unnamed: 11': 'undesirable/desirable',
    'Unnamed: 12': 'raising alertness/sleep-inducing',
    'I trust that the autonomous vehicle will operate safely and effectively under various conditions [Please click one that applies]': 'trust-safety-effectiveness'
}, inplace=True)

print(df.columns)

Index(['participant_id', 'experiment_condition', 'personality_trait',
       'useful/useless', 'pleasant/unpleasant', 'bad/good', 'nice/annoying',
       'effective/superfluous', 'irritating/likeable', 'assisting/worthless',
       'undesirable/desirable', 'raising alertness/sleep-inducing',
       'trust-safety-effectiveness'],
      dtype='object')


In [ ]:
df

,participant_id,experiment_condition,personality_trait,useful/useless,pleasant/unpleasant,bad/good,nice/annoying,effective/superfluous,irritating/likeable,assisting/worthless,undesirable/desirable,raising alertness/sleep-inducing,trust-safety-effectiveness
0,1,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,2,Agree
1,1,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,3,Agree
2,2,A,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree
3,2,C,High Openness - Low Neuroticism,1,1,5,1,1,5,1,5,1,Agree
4,3,A,High Neuroticism - Low Openness,1,2,4,2,2,5,2,5,2,Agree
5,3,B,High Neuroticism - Low Openness,1,2,5,2,2,4,2,4,2,Agree
6,4,A,High Neuroticism - Low Openness,3,3,3,2,3,4,1,4,1,Agree
7,4,C,High Neuroticism - Low Openness,3,2,3,3,3,4,2,3,3,Agree
8,5,A,High Neuroticism - Low Openness,3,3,3,3,2,2,3,3,3,Neutral
9,5,B,High Neuroticism - Low Openness,3,4,3,3,4,2,4,3,3,Neutral


In [ ]:
import pandas as pd

# Assuming df is your DataFrame

# Update the map_score function to handle both kinds of mappings: from 1-5 scale and response strings
def map_score(score):
    if isinstance(score, int):  # Handle the 1-5 scale mapping
        mapping = {1: 2, 2: 1, 3: 0, 4: -1, 5: -2}
        return mapping[score]
    elif isinstance(score, str):  # Handle response string mapping
        mapping = {
            'Strongly Disagree': -2.0,
            'Disagree': -1.0,
            'Neutral': 0.0,
            'Agree': 1.0,
            'Strongly Agree': 2.0
        }
        return mapping[score]
    return None

# Apply the mapping to all relevant columns
score_columns = [
    'useful/useless', 'bad/good', 'effective/superfluous', 'assisting/worthless',
    'raising alertness/sleep-inducing', 'pleasant/unpleasant', 'nice/annoying',
    'irritating/likeable', 'undesirable/desirable', 'trust-safety-effectiveness'
]
for col in score_columns:
    df[col] = df[col].apply(map_score)

# Define the function to calculate Usefulness and Satisfying scores
def calculate_scores(row):
    # Calculating Usefulness
    usefulness = (
        row['useful/useless'] +
        (-row['bad/good']) +  # Mirrored
        row['effective/superfluous'] +
        row['assisting/worthless'] +
        row['raising alertness/sleep-inducing']
    ) / 5

    # Calculating Satisfying
    satisfying = (
        row['pleasant/unpleasant'] +
        row['nice/annoying'] +  # Mirrored
        (-row['irritating/likeable']) +  # Mirrored
        (-row['undesirable/desirable'])  # Mirrored
    ) / 4

    return pd.Series([usefulness, satisfying])

# Calculate Usefulness and Satisfying scores
df[['Usefulness', 'Satisfying']] = df.apply(calculate_scores, axis=1)

# Add Trust-Safety score to the DataFrame
df['Trust-Safety'] = df['trust-safety-effectiveness']

# Create a new DataFrame with the necessary columns
result_df = df[['participant_id', 'experiment_condition', 'Usefulness', 'Satisfying', 'Trust-Safety']]

# Display the updated DataFrame
print(result_df)


    participant_id experiment_condition  Usefulness  Satisfying  Trust-Safety
0                1                    A         1.8        2.00           1.0
1                1                    C         1.6        2.00           1.0
2                2                    A         2.0        2.00           1.0
3                2                    C         2.0        2.00           1.0
4                3                    A         1.2        1.50           1.0
5                3                    B         1.4        1.00           1.0
6                4                    A         0.8        0.75           1.0
7                4                    C         0.2        0.50           1.0
8                5                    A         0.2       -0.25           0.0
9                5                    B        -0.4       -0.50           0.0
10               6                    A         0.4        0.00          -2.0
11               6                    C        -0.4       -0.75 

In [ ]:
result_df

,participant_id,experiment_condition,Usefulness,Satisfying,Trust-Safety
0,1,A,1.8,2.00,1.0
1,1,C,1.6,2.00,1.0
2,2,A,2.0,2.00,1.0
3,2,C,2.0,2.00,1.0
4,3,A,1.2,1.50,1.0
5,3,B,1.4,1.00,1.0
6,4,A,0.8,0.75,1.0
7,4,C,0.2,0.50,1.0
8,5,A,0.2,-0.25,0.0
9,5,B,-0.4,-0.50,0.0


In [ ]:
def assign_condition_number(row):
    # Map each condition to a number
    mapping = {'A': 0, 'B': 1, 'C': 2}
    return mapping.get(row['experiment_condition'], -1)  # Returns -1 if none of A, B, C

# Apply the function to create the new column
result_df['condition_number'] = result_df.apply(assign_condition_number, axis=1)

<ipython-input-6-9e60c9b5142f>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  result_df['condition_number'] = result_df.apply(assign_condition_number, axis=1)


In [ ]:
result_df

,participant_id,experiment_condition,Usefulness,Satisfying,Trust-Safety,condition_number
0,1,A,1.8,2.00,1.0,0
1,1,C,1.6,2.00,1.0,2
2,2,A,2.0,2.00,1.0,0
3,2,C,2.0,2.00,1.0,2
4,3,A,1.2,1.50,1.0,0
5,3,B,1.4,1.00,1.0,1
6,4,A,0.8,0.75,1.0,0
7,4,C,0.2,0.50,1.0,2
8,5,A,0.2,-0.25,0.0,0
9,5,B,-0.4,-0.50,0.0,1


In [ ]:
result_df.to_excel(
  'acceptance_results.xlsx'
)